In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project root
PROJECT_ROOT = Path.cwd().parent

# Dataset paths
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_TRANSACTION = DATA_DIR / "train_transaction.csv"
TRAIN_IDENTITY = DATA_DIR / "train_identity.csv"
TEST_TRANSACTION = DATA_DIR / "test_transaction.csv"
TEST_IDENTITY = DATA_DIR / "test_identity.csv"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Train transaction exists:", TRAIN_TRANSACTION.exists())
print("Train identity exists:", TRAIN_IDENTITY.exists())

Project root: /Users/ayushkumar/Desktop/Fraudguard
Data directory: /Users/ayushkumar/Desktop/Fraudguard/data/raw
Train transaction exists: True
Train identity exists: True


In [3]:
# Inspect the first 5 rows of the training transaction dataset
train_sample = pd.read_csv(
    TRAIN_TRANSACTION,
    nrows=5
)

print("Rows loaded:", len(train_sample))
print("Number of columns:", len(train_sample.columns))

print("\nColumn names:")
print(train_sample.columns.tolist())

Rows loaded: 5
Number of columns: 394

Column names:
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74'

In [4]:
# Inspect data types and basic structure

print("Data Types:")
print(train_sample.dtypes.value_counts())

print("\nFirst 5 rows:")
display(train_sample.head())

print("\nDataset Info:")
train_sample.info()

Data Types:
float64    377
object      13
int64        4
Name: count, dtype: int64

First 5 rows:


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Columns: 394 entries, TransactionID to V339
dtypes: float64(377), int64(4), object(13)
memory usage: 15.5+ KB


In [5]:
# Inspect the target variable

print("Target column:", "isFraud")

print("\nTarget values:")
print(train_sample["isFraud"].value_counts())

print("\nTarget percentages:")
print(train_sample["isFraud"].value_counts(normalize=True) * 100)

Target column: isFraud

Target values:
isFraud
0    5
Name: count, dtype: int64

Target percentages:
isFraud
0    100.0
Name: proportion, dtype: float64


In [6]:
# Calculate the actual fraud distribution using chunks
# This avoids loading the entire 658 MB dataset into memory.

fraud_counts = {0: 0, 1: 0}

for chunk in pd.read_csv(
    TRAIN_TRANSACTION,
    usecols=["isFraud"],
    chunksize=100_000
):
    counts = chunk["isFraud"].value_counts()
    
    fraud_counts[0] += counts.get(0, 0)
    fraud_counts[1] += counts.get(1, 0)

total_transactions = fraud_counts[0] + fraud_counts[1]

print("Total transactions:", f"{total_transactions:,}")
print("Legitimate transactions:", f"{fraud_counts[0]:,}")
print("Fraudulent transactions:", f"{fraud_counts[1]:,}")

print("\nFraud distribution:")
print(f"Legitimate: {fraud_counts[0] / total_transactions * 100:.4f}%")
print(f"Fraud:      {fraud_counts[1] / total_transactions * 100:.4f}%")

Total transactions: 590,540
Legitimate transactions: 569,877
Fraudulent transactions: 20,663

Fraud distribution:
Legitimate: 96.5010%
Fraud:      3.4990%


In [7]:
# Inspect the identity dataset

identity_sample = pd.read_csv(
    TRAIN_IDENTITY,
    nrows=5
)

print("Rows loaded:", len(identity_sample))
print("Number of columns:", len(identity_sample.columns))

print("\nColumn names:")
print(identity_sample.columns.tolist())

print("\nData types:")
print(identity_sample.dtypes.value_counts())

Rows loaded: 5
Number of columns: 41

Column names:
['TransactionID', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_07', 'id_08', 'id_09', 'id_10', 'id_11', 'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']

Data types:
float64    25
object     15
int64       1
Name: count, dtype: int64


In [8]:
# Count rows in the identity dataset efficiently

identity_rows = 0

for chunk in pd.read_csv(
    TRAIN_IDENTITY,
    usecols=["TransactionID"],
    chunksize=100_000
):
    identity_rows += len(chunk)

print("Total identity records:", f"{identity_rows:,}")


Total identity records: 144,233


In [10]:
# Analyze missing values in the transaction dataset
# We process the file in chunks to avoid loading the entire dataset into memory.

missing_counts = None
total_rows = 0

for chunk in pd.read_csv(
    TRAIN_TRANSACTION,
    chunksize=100_000
):
    total_rows += len(chunk)

    chunk_missing = chunk.isna().sum()

    if missing_counts is None:
        missing_counts = chunk_missing
    else:
        missing_counts += chunk_missing

missing_percentage = (missing_counts / total_rows) * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percentage": missing_percentage
})

missing_summary = missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_summary.head(30))

,missing_count,missing_percentage
dist2,552913,93.628374
D7,551623,93.409930
D13,528588,89.509263
D14,528353,89.469469
D12,525823,89.041047
D6,517353,87.606767
D9,515614,87.312290
D8,515614,87.312290
V153,508595,86.123717
V139,508595,86.123717


In [11]:
# Investigate whether missingness itself is related to fraud

# Read only the required columns
d7_fraud = pd.read_csv(
    TRAIN_TRANSACTION,
    usecols=["D7", "isFraud"]
)

d7_fraud["D7_missing"] = d7_fraud["D7"].isna()

missing_fraud_rate = (
    d7_fraud.groupby("D7_missing")["isFraud"]
    .agg(["count", "sum", "mean"])
)

missing_fraud_rate["fraud_percentage"] = (
    missing_fraud_rate["mean"] * 100
)

missing_fraud_rate.index = [
    "D7 Present",
    "D7 Missing"
]

display(missing_fraud_rate)

,count,sum,mean,fraud_percentage
D7 Present,38917,5790,0.148778,14.877817
D7 Missing,551623,14873,0.026962,2.696226


In [12]:
# Categorize features by their missing-value percentage

missing_summary["missing_category"] = pd.cut(
    missing_summary["missing_percentage"],
    bins=[-1, 20, 50, 80, 100],
    labels=[
        "Low (<20%)",
        "Moderate (20–50%)",
        "High (50–80%)",
        "Very High (>80%)"
    ]
)

print("Features by missingness category:\n")

print(
    missing_summary["missing_category"]
    .value_counts()
    .sort_index()
)

Features by missingness category:

missing_category
Low (<20%)           182
Moderate (20–50%)     38
High (50–80%)        119
Very High (>80%)      55
Name: count, dtype: int64


In [13]:
# Identify categorical/object features

categorical_features = train_sample.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical features:", len(categorical_features))
print()

for feature in categorical_features:
    print(feature)
    

Categorical features: 13

ProductCD
card4
card6
P_emaildomain
M1
M2
M3
M4
M5
M6
M7
M8
M9


In [14]:
# Analyze categorical feature cardinality

categorical_cardinality = []

for feature in categorical_features:
    unique_count = train_sample[feature].nunique(dropna=True)
    
    categorical_cardinality.append({
        "feature": feature,
        "unique_values_in_sample": unique_count
    })

cardinality_df = pd.DataFrame(categorical_cardinality)

display(cardinality_df)


,feature,unique_values_in_sample
0,ProductCD,2
1,card4,3
2,card6,2
3,P_emaildomain,3
4,M1,1
5,M2,1
6,M3,1
7,M4,2
8,M5,2
9,M6,2


In [15]:
# Calculate actual categorical cardinality across the full dataset

categorical_cardinality = {}

for chunk in pd.read_csv(
    TRAIN_TRANSACTION,
    usecols=categorical_features,
    chunksize=100_000
):
    for feature in categorical_features:
        values = chunk[feature].dropna().unique()
        
        if feature not in categorical_cardinality:
            categorical_cardinality[feature] = set()
        
        categorical_cardinality[feature].update(values)

cardinality_df = pd.DataFrame({
    "feature": list(categorical_cardinality.keys()),
    "unique_values": [
        len(values)
        for values in categorical_cardinality.values()
    ]
})

cardinality_df = cardinality_df.sort_values(
    "unique_values",
    ascending=False
)

display(cardinality_df)

,feature,unique_values
3,P_emaildomain,59
0,ProductCD,5
1,card4,4
2,card6,4
7,M4,3
4,M1,2
5,M2,2
6,M3,2
8,M5,2
9,M6,2


In [16]:
# Analyze missing values in the identity dataset

identity_missing_counts = None
identity_total_rows = 0

for chunk in pd.read_csv(
    TRAIN_IDENTITY,
    chunksize=100_000
):
    identity_total_rows += len(chunk)

    chunk_missing = chunk.isna().sum()

    if identity_missing_counts is None:
        identity_missing_counts = chunk_missing
    else:
        identity_missing_counts += chunk_missing

identity_missing_percentage = (
    identity_missing_counts / identity_total_rows
) * 100

identity_missing_summary = pd.DataFrame({
    "missing_count": identity_missing_counts,
    "missing_percentage": identity_missing_percentage
})

identity_missing_summary = identity_missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

display(identity_missing_summary)


,missing_count,missing_percentage
id_24,139486,96.708798
id_25,139101,96.441868
id_07,139078,96.425922
id_08,139078,96.425922
id_21,139074,96.423149
id_26,139070,96.420375
id_23,139064,96.416215
id_27,139064,96.416215
id_22,139064,96.416215
id_18,99120,68.722137


In [17]:
# Identify categorical features in the identity dataset

identity_sample = pd.read_csv(
    TRAIN_IDENTITY,
    nrows=1000
)

identity_categorical_features = identity_sample.select_dtypes(
    include=["object"]
).columns.tolist()

print("Identity categorical features:", len(identity_categorical_features))
print()

for feature in identity_categorical_features:
    print(feature)

Identity categorical features: 17

id_12
id_15
id_16
id_23
id_27
id_28
id_29
id_30
id_31
id_33
id_34
id_35
id_36
id_37
id_38
DeviceType
DeviceInfo


In [18]:
# Calculate actual categorical cardinality for the identity dataset

identity_categorical_cardinality = {}

for chunk in pd.read_csv(
    TRAIN_IDENTITY,
    usecols=identity_categorical_features,
    chunksize=50_000
):
    for feature in identity_categorical_features:
        values = chunk[feature].dropna().unique()

        if feature not in identity_categorical_cardinality:
            identity_categorical_cardinality[feature] = set()

        identity_categorical_cardinality[feature].update(values)

identity_cardinality_df = pd.DataFrame({
    "feature": list(identity_categorical_cardinality.keys()),
    "unique_values": [
        len(values)
        for values in identity_categorical_cardinality.values()
    ]
})

identity_cardinality_df = identity_cardinality_df.sort_values(
    "unique_values",
    ascending=False
)

display(identity_cardinality_df)

,feature,unique_values
16,DeviceInfo,1786
9,id_33,260
8,id_31,130
7,id_30,75
10,id_34,4
3,id_23,3
1,id_15,3
4,id_27,2
5,id_28,2
6,id_29,2


In [19]:
# Check TransactionID uniqueness in the training transaction dataset

transaction_ids = pd.read_csv(
    TRAIN_TRANSACTION,
    usecols=["TransactionID"]
)

total_ids = len(transaction_ids)
unique_ids = transaction_ids["TransactionID"].nunique()

print("Total TransactionIDs:", f"{total_ids:,}")
print("Unique TransactionIDs:", f"{unique_ids:,}")
print("Duplicate TransactionIDs:", f"{total_ids - unique_ids:,}")

Total TransactionIDs: 590,540
Unique TransactionIDs: 590,540
Duplicate TransactionIDs: 0


In [20]:
# Check TransactionID uniqueness in the identity dataset

identity_ids = pd.read_csv(
    TRAIN_IDENTITY,
    usecols=["TransactionID"]
)

total_identity_ids = len(identity_ids)
unique_identity_ids = identity_ids["TransactionID"].nunique()

print("Total identity TransactionIDs:", f"{total_identity_ids:,}")
print("Unique identity TransactionIDs:", f"{unique_identity_ids:,}")
print(
    "Duplicate identity TransactionIDs:",
    f"{total_identity_ids - unique_identity_ids:,}"
)

Total identity TransactionIDs: 144,233
Unique identity TransactionIDs: 144,233
Duplicate identity TransactionIDs: 0


In [21]:
# Verify identity-to-transaction join coverage

transaction_ids_set = set(
    pd.read_csv(
        TRAIN_TRANSACTION,
        usecols=["TransactionID"]
    )["TransactionID"]
)

identity_ids = pd.read_csv(
    TRAIN_IDENTITY,
    usecols=["TransactionID"]
)["TransactionID"]

matched = identity_ids.isin(transaction_ids_set).sum()
unmatched = len(identity_ids) - matched

print("Identity records:", f"{len(identity_ids):,}")
print("Matched transactions:", f"{matched:,}")
print("Unmatched identity records:", f"{unmatched:,}")
print("Match rate:", f"{matched / len(identity_ids) * 100:.4f}%")

Identity records: 144,233
Matched transactions: 144,233
Unmatched identity records: 0
Match rate: 100.0000%
